# 🤖 Notebook 5: Model Training

We train four models representing a progression from simple to complex:

| Model | Type | Key Strength |
|---|---|---|
| Logistic Regression | Linear | Interpretable baseline |
| Random Forest | Ensemble (bagging) | Handles nonlinearity, robust |
| XGBoost | Ensemble (boosting) | High accuracy, good with imbalance |
| LightGBM | Ensemble (boosting) | Fastest, best accuracy on tabular data |


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import pickle, os, time, warnings
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
import sys
sys.path.insert(0, '..')
from src.data_loader import load_data
from src.feature_engineering import engineer_features, get_feature_names
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.facecolor': '#0f1117', 'axes.facecolor': '#0f1117',
    'text.color': 'white', 'axes.labelcolor': 'white',
    'xtick.color': 'white', 'ytick.color': 'white',
    'axes.edgecolor': '#444',
})

DATA_PATH = '../data/PS_20174392719_1491204439457_log.csv'
os.makedirs('../models', exist_ok=True)

# Load & prepare
df = load_data(DATA_PATH, sample_frac=0.2)
df_fe = engineer_features(df)
feature_cols = [c for c in get_feature_names(df_fe) if c in df_fe.columns]
X = df_fe[feature_cols].fillna(0)
y = df_fe['isFraud']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

# SMOTE
sm = SMOTE(sampling_strategy=0.1, random_state=42, n_jobs=-1)
X_train_sm, y_train_sm = sm.fit_resample(X_train, y_train)

# Scale for LR
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train_sm)
X_test_sc = scaler.transform(X_test)
pickle.dump(scaler, open('../models/scaler.pkl', 'wb'))

print("Data prepared ✓")
print(f"Train: {len(X_train_sm):,} | Test: {len(X_test):,}")


In [ ]:
# ── Model 1: Logistic Regression (Baseline) ──────────────────────────────
print("Training Logistic Regression...")
t0 = time.time()
lr = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42, n_jobs=-1)
lr.fit(X_train_sc, y_train_sm)
lr_time = time.time() - t0

lr_prob = lr.predict_proba(X_test_sc)[:, 1]
lr_auc = roc_auc_score(y_test, lr_prob)
print(f"  ROC-AUC: {lr_auc:.4f}  |  Time: {lr_time:.1f}s")
pickle.dump(lr, open('../models/logistic_regression.pkl', 'wb'))


In [ ]:
# ── Model 2: Random Forest ────────────────────────────────────────────────
print("Training Random Forest...")
t0 = time.time()
rf = RandomForestClassifier(
    n_estimators=200, max_depth=15, min_samples_leaf=10,
    class_weight='balanced', random_state=42, n_jobs=-1
)
rf.fit(X_train_sm, y_train_sm)
rf_time = time.time() - t0

rf_prob = rf.predict_proba(X_test)[:, 1]
rf_auc = roc_auc_score(y_test, rf_prob)
print(f"  ROC-AUC: {rf_auc:.4f}  |  Time: {rf_time:.1f}s")
pickle.dump(rf, open('../models/random_forest.pkl', 'wb'))


In [ ]:
# ── Model 3: XGBoost ──────────────────────────────────────────────────────
print("Training XGBoost...")
t0 = time.time()
n_neg = int((y_train_sm == 0).sum())
n_pos = int((y_train_sm == 1).sum())

xgb = XGBClassifier(
    n_estimators=300, max_depth=7, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8,
    scale_pos_weight=n_neg/n_pos,
    eval_metric='auc', random_state=42, n_jobs=-1, verbosity=0
)
xgb.fit(X_train_sm, y_train_sm)
xgb_time = time.time() - t0

xgb_prob = xgb.predict_proba(X_test)[:, 1]
xgb_auc = roc_auc_score(y_test, xgb_prob)
print(f"  ROC-AUC: {xgb_auc:.4f}  |  Time: {xgb_time:.1f}s")
pickle.dump(xgb, open('../models/xgboost.pkl', 'wb'))


In [ ]:
# ── Model 4: LightGBM ─────────────────────────────────────────────────────
print("Training LightGBM...")
t0 = time.time()
lgbm = LGBMClassifier(
    n_estimators=500, max_depth=8, learning_rate=0.05,
    num_leaves=63, subsample=0.8, colsample_bytree=0.8,
    class_weight='balanced', random_state=42, n_jobs=-1, verbose=-1
)
lgbm.fit(X_train_sm, y_train_sm)
lgbm_time = time.time() - t0

lgbm_prob = lgbm.predict_proba(X_test)[:, 1]
lgbm_auc = roc_auc_score(y_test, lgbm_prob)
print(f"  ROC-AUC: {lgbm_auc:.4f}  |  Time: {lgbm_time:.1f}s")
pickle.dump(lgbm, open('../models/lightgbm.pkl', 'wb'))


In [ ]:
# ── Summary Table ─────────────────────────────────────────────────────────
results = pd.DataFrame({
    'Model': ['Logistic Regression', 'Random Forest', 'XGBoost', 'LightGBM'],
    'ROC_AUC': [lr_auc, rf_auc, xgb_auc, lgbm_auc],
    'Training_Time_s': [lr_time, rf_time, xgb_time, lgbm_time]
})
results = results.sort_values('ROC_AUC', ascending=False)
results['ROC_AUC'] = results['ROC_AUC'].round(4)
results['Training_Time_s'] = results['Training_Time_s'].round(1)

os.makedirs('../reports', exist_ok=True)
results.to_csv('../reports/model_comparison.csv', index=False)

print("\n" + "="*50)
print("  MODEL COMPARISON RESULTS")
print("="*50)
print(results.to_string(index=False))
print(f"\n🏆 Best: {results.iloc[0]['Model']} (ROC-AUC = {results.iloc[0]['ROC_AUC']:.4f})")

# Save probabilities for evaluation notebook
probs = {
    'Logistic Regression': lr_prob,
    'Random Forest': rf_prob,
    'XGBoost': xgb_prob,
    'LightGBM': lgbm_prob
}
pickle.dump({'probs': probs, 'y_test': y_test, 'X_test': X_test,
             'feature_cols': feature_cols}, open('../models/eval_data.pkl', 'wb'))
print("\n✅ All models and evaluation data saved.")
